# Layer 5-7: Backtest Deep Dive Template

This notebook demonstrates an ADR-017 strategy deep dive workflow using the modern DataLoader + VectorizedBacktester stack.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    target = Path('scripts/trading_framework/config/sessions.yaml')
    for p in [start, *start.parents]:
        if (p / target).exists():
            return p
    raise FileNotFoundError('Could not locate repo root containing sessions.yaml')

ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
CONFIG_PATH = ROOT / 'scripts' / 'trading_framework' / 'config' / 'sessions.yaml'

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from scripts.trading_framework.config.config_loader import load_config
from scripts.libs.data.loader import DataLoader
from scripts.strategies.reversal.core.box_reversion import BoxReversionStrategy
from scripts.trading_framework.core.backtest_engine import VectorizedBacktester
from scripts.trading_framework.reporting.reporter import QuantReporter

print('Repo root:', ROOT)
print('CWD:', Path.cwd())
print('Config path:', CONFIG_PATH)

## 1. Load Data (Layer 1)

In [ ]:
ticker = 'NQ1'
cfg = load_config(str(CONFIG_PATH))
loader = DataLoader(cfg)
df = loader.load_enriched(ticker)
df.tail()

## 2. Generate Signals (Layer 4)

In [ ]:
strategy = BoxReversionStrategy(ticker=ticker)
config = {
    'filter_high_vol': True,
    'min_dist': 0.0005,
    'sl_dist': 0.0050,
}

signals = strategy.hunt(df.last('30D'), config)
signals.head(), len(signals)

## 3. Run Vectorized Engine (Layer 5)

In [ ]:
engine = VectorizedBacktester()
metrics = engine.run(signals, df.last('30D'))
print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
print(f"Max drawdown: {metrics['max_drawdown_%']:.2f}%")

## 4. Institutional Reporting (Layer 7 + ADR-010)

In [ ]:
from datetime import datetime
run_id = f"RESEARCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{ticker}_DeepDive"
reporter = QuantReporter(run_id=run_id)

returns = metrics['equity_curve'].pct_change().fillna(0)
reporter.generate_tear_sheet(returns, "Research_TearSheet")
print(f"Results stored in: {reporter.output_dir}")